# 03 — Consensus UMAP: Methodology Details

This notebook documents the consensus UMAP methodology in detail:
1. Why consensus? (UMAP instability across seeds)
2. Distance-matrix averaging vs. coordinate averaging
3. ARI comparison (0.71 vs 0.56)
4. Projection head validation (R² = 0.73, k-NN = 82.4%)

This is the methodological contribution companion to `supplementary/consensus_umap_implementation.md`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.consensus_umap import (
    run_umap_multi_seed, procrustes_align, compute_consensus_average,
    distance_matrix_consensus, umap_from_precomputed_distances,
    compute_pairwise_ari, l2_normalize,
)
from src.clustering import run_kmeans
from src.models import UMAPConfig

## 1. The UMAP Stability Problem

UMAP is a stochastic algorithm — different random seeds produce different embeddings.
For cross-corpus comparison, we need a *stable* reference geometry.

We solve this with **consensus UMAP**: run UMAP with 31 different seeds,
then combine the results into a single stable embedding.

## 2. Two Consensus Approaches

### Approach A: Coordinate Averaging (Procrustes)
1. Run UMAP 31 times → 31 embeddings
2. Align via orthogonal Procrustes (rotate each to match the first)
3. Average the aligned coordinates

**Problem:** Procrustes alignment preserves global rotation but not local structure.
Mean ARI (seed vs. consensus): **0.56**

### Approach B: Distance-Matrix Averaging (Recommended)
1. Run UMAP 31 times → 31 embeddings
2. Compute pairwise distance matrix for each
3. Average the distance matrices
4. Run UMAP once on the averaged distance matrix (metric='precomputed')

**Advantage:** Distances are rotation-invariant. No alignment needed.
Mean ARI (seed vs. consensus): **0.71**

We use Approach B throughout the paper.

## 3. Projection Head Architecture

To project new data (artist probes, public probes) into the consensus space:

- **Model:** MLPRegressor (scikit-learn)
- **Architecture:** 1024 → 512 → 128 → 8
- **Activation:** ReLU
- **Training:** StandardScaler on both input and output, early stopping
- **Validation:** R² = 0.73, k-NN neighborhood preservation = 82.4%

The projection head learns the mapping from high-dimensional embedding space
to the stable consensus coordinate system, enabling cross-corpus comparison
without distorting the reference map.